In [1]:
import pandas as pd
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay

reports_dir = Path("../reports/baseline")
reports_dir.mkdir(parents=True, exist_ok=True)

# 1. Загрузка подготовленных выборок и препроцессора
data_dir = Path("../data/model")
X_train = pd.read_csv(data_dir / "X_train.csv")
X_val = pd.read_csv(data_dir / "X_validation.csv")
y_train = pd.read_csv(data_dir / "y_train.csv").squeeze("columns")
y_val = pd.read_csv(data_dir / "y_validation.csv").squeeze("columns")

preprocessor = joblib.load("../models/feature_preprocessor.joblib")

# 2. Создание и оценка наивной модели (Dummy)
dummy_model = DummyClassifier(strategy="most_frequent", random_state=42)
dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_val)

dummy_f1 = f1_score(y_val, dummy_predictions, average="weighted", zero_division=0)
print(f"F1-score наивной модели: {dummy_f1:.4f}")

# 3. Разработка базовой модели (Логистическая регрессия)
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# Объединяем препроцессор и модель в единый Pipeline
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", log_reg)
])

# Обучаем базовую модель
baseline_model.fit(X_train, y_train)
val_predictions = baseline_model.predict(X_val)

# 4. Расчет метрик базовой модели
acc = accuracy_score(y_val, val_predictions)
prec = precision_score(y_val, val_predictions, average="weighted", zero_division=0)
rec = recall_score(y_val, val_predictions, average="weighted", zero_division=0)
f1 = f1_score(y_val, val_predictions, average="weighted", zero_division=0)

print("\nМетрики базовой модели (Логистическая регрессия):")
print(f"Accuracy: {acc:.4f}\nPrecision: {prec:.4f}\nRecall: {rec:.4f}\nF1-score: {f1:.4f}")

# 5. Проверка на переобучение
train_predictions = baseline_model.predict(X_train)
train_f1 = f1_score(y_train, train_predictions, average="weighted", zero_division=0)
print(f"\nF1 на обучении: {train_f1:.4f} | F1 на валидации: {f1:.4f}")
print("Разница невелика, значит переобучения нет.")

# 6. Матрица ошибок (визуализация)
ConfusionMatrixDisplay.from_predictions(y_val, val_predictions)
plt.title("Матрица ошибок базовой модели")
plt.tight_layout()
plt.savefig(reports_dir / "confusion_matrix.png", dpi=150)
plt.close()
print(f"\nМатрица ошибок сохранена в {reports_dir}")

# 7. Сохранение базовой модели и метрик
joblib.dump(baseline_model, "../models/baseline_model.joblib")

metrics_df = pd.DataFrame({
    "Метрика": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Значение": [acc, prec, rec, f1]
})
metrics_df.to_csv(reports_dir / "baseline_metrics.csv", index=False, encoding="utf-8-sig")
print("Базовая модель и метрики успешно сохранены.")

F1-score наивной модели: 0.3333

Метрики базовой модели (Логистическая регрессия):
Accuracy: 0.5500
Precision: 0.5549
Recall: 0.5500
F1-score: 0.5396

F1 на обучении: 0.5372 | F1 на валидации: 0.5396
Разница невелика, значит переобучения нет.

Матрица ошибок сохранена в ..\reports\baseline
Базовая модель и метрики успешно сохранены.
